# Build A Basic Chatbot With Langgraph(GRAPH API)

In [ ]:
from typing import Annotated

from typing_extensions import TypedDict

from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages

In [ ]:
class State(TypedDict):
    # Messages have the type "list". The `add_messages` function
    # in the annotation defines how this state key should be updated
    # (in this case, it appends messages to the list, rather than overwriting them)
    messages:Annotated[list,add_messages]

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model

llm=ChatGroq(model="llama-3.1-8b-instant")

In [ ]:
## Node Functionality
def chatbot(state:State):
    return {"messages":[llm.invoke(state["messages"])]}

In [ ]:
graph_builder = StateGraph(State)

#Adding node
graph_builder.add_node("llmchatbot", chatbot)

#Adding edges
graph_builder.add_edge(START, "llmchatbot")
graph_builder.add_edge("llmchatbot", END )

#compile Graph
graph = graph_builder.compile()

In [ ]:
#Visualize the graph
from IPython.display import Image,display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    pass

In [ ]:
response = graph.invoke({"messages":"Hi"}) # type: ignore

In [ ]:
response["messages"][-1].content

In [ ]:
for event in graph.stream({"messages":"Hi, How are you?" }):
    for value in event.values():
        print(value["messages"][-1].content)

## Chatbot With Tool

In [ ]:
from langchain_tavily import TavilySearch

tool=TavilySearch(max_results=2)
tool.invoke("What is langgraph")

In [ ]:
#Custom function
def multiply(a:int,b:int)->int:
    """Multiply a and b

    Args:
        a (int): first int
        b (int): second int

    Returns:
        int: output int
    """

    return a*b

In [ ]:
tools =[tool, multiply]

In [ ]:
llm_with_tool = llm.bind_tools(tools)

In [ ]:
llm_with_tool

In [ ]:
##StateGraph

from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

##Node Definition
def tool_calling_llm(state:State):
    return{"messages":[llm_with_tool.invoke(state["messages"])]}

##Graph
builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)  # Add the function reference
builder.add_node("tools", ToolNode(tools))

##Add Edges
builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    tools_condition,
    
)

builder.add_edge("tools", END)

##Compile the graph
graph = builder.compile()

from IPython.display import Image,display
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
response = graph.invoke({"messages":"What is the most recent AI news"}) #type:ignore

In [ ]:
for m in response['messages']:
    m.pretty_print()

In [ ]:
response = graph.invoke({"messages":"What is 92 multiplied by 34 and then multiply 5"}) #type:ignore

for m in response['messages']:
    m.pretty_print()

## ReAct Agent Architecture

In [ ]:
##StateGraph

from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

##Node Definition
def tool_calling_llm(state:State):
    return{"messages":[llm_with_tool.invoke(state["messages"])]}

##Graph
builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)  # Add the function reference
builder.add_node("tools", ToolNode(tools))

##Add Edges
builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    tools_condition,
    
)

builder.add_edge("tools", "tool_calling_llm")

##Compile the graph
graph = builder.compile()

from IPython.display import Image,display
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
response = graph.invoke({"messages":"What is 92 multiplied by 34 and what's the latest AI news?"}) #type:ignore

for m in response['messages']:
    m.pretty_print()

## Adding Memory in Agentic Graph

In [ ]:
##StateGraph

from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

##Node Definition
def tool_calling_llm(state:State):
    return{"messages":[llm_with_tool.invoke(state["messages"])]}

##Graph
builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)  # Add the function reference
builder.add_node("tools", ToolNode(tools))

##Add Edges
builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    tools_condition,
    
)

builder.add_edge("tools", END)

##Compile the graph
graph = builder.compile(checkpointer= memory)

from IPython.display import Image,display
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
config = {"configurable":{"thread_id":"1"}}

response = graph.invoke({"messages":"Hi my name is Shivam"}, config=config) #type:ignore

response

In [ ]:
response['messages'][-1].content

In [ ]:
response = graph.invoke({"messages":"Hi what is my name?"}, config=config) #type:ignore

response['messages'][-1].content

In [ ]:
response = graph.invoke({"messages":"Hey do you remember my name?"}, config=config) #type:ignore

response['messages'][-1].content

In [ ]:
def superbot(state:State):
    return{"messages": llm.invoke(state['messages'])}

In [ ]:
graph = StateGraph(State)

#node
graph.add_node("SuperBot", superbot)

#Edges
graph.add_edge(START, "SuperBot")
graph.add_edge("SuperBot", END)

graph_builder = graph.compile(checkpointer= memory)

from IPython.display import display, Image
display(Image(graph_builder.get_graph().draw_mermaid_png()))

In [ ]:
#Invocation

config = {"configurable": {"thread_id":"1"}}

graph_builder.invoke({"messages": "Hi my is Shivam & I like football"},config) #type:ignore

### Streaming

Methods .stream() and astream()

- These methods are sync and async methods for streaming back results.

Additional parameters in streaming nodes for graph state

- **values** : This streams the full state of the graph aftwer each node is called.
- **updates** : This streams updates to the state of the graph aftyer each node is called.
    

In [ ]:
#Create node

config =  {"configurable":{"thread_id":"3"}}

for chunk in graph_builder.stream({"messages":"Hi my name is Shivam and I love football"}, config, stream_mode= "updates"):
    print(chunk)

In [ ]:
for chunk in graph_builder.stream({"messages":"Hi my name is Shivam and I love football"}, config, stream_mode= "values"):
    print(chunk)

In [ ]:
#Create node

config =  {"configurable":{"thread_id":"4"}}

for chunk in graph_builder.stream({"messages":"Hi my name is Shivam and I love football"}, config, stream_mode= "updates"):
    print(chunk)

In [ ]:
for chunk in graph_builder.stream({"messages":"I also like badminton"}, config, stream_mode= "values"):
    print(chunk)

In [ ]:
#Create node

config =  {"configurable":{"thread_id":"5"}}

async for event in graph_builder.astream_events({"messages":"Hi my name is Shivam and I love football"}, config, version= "v2"):
    print(event)